In [ ]:
# ===================== CELLULE 1 — Config environnement GPU =====================
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
# ===================== CELLULE 2 — Imports =====================
import os, sys, time, json, datetime, argparse
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A

# Acces au schema NPZ partage du projet
PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified_npz, load_unified


In [ ]:
# ===================== CELLULE 3 — Configuration (TrainConfig) =====================
from dataclasses import dataclass, field

@dataclass
class TrainConfig:
    # Configuration d'entrainement pour la detection de points d'intersection (5mm) via NPZ.

    # Chemins
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    npz_dir:   str = ""
    output_dir: str = ""

    # Cible : intersections de la grille 5mm (cle NPZ)
    npz_key: str = "grid_major_5mm"
    point_radius: int = 5         # rayon (px) de la fenetre dans laquelle la gaussienne est dessinee
    gaussian_sigma: float = 2.0   # ecart-type de la gaussienne 2D (px). Pic au centre = 1.0, decroit vers les bords.

    # Architecture
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    # Resolution
    img_height: int = 1024
    img_width:  int = 1024

    # Entrainement
    batch_size: int = 4
    num_epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-5
    num_workers: int = 0
    pin_memory:  bool = True

    # Loss & scheduler
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # Split par prefixe de nom de fichier
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources:   list = field(default_factory=lambda: ["ECG_033"])

    device: str = ""
    seed: int = 42
    save_every_n_epochs:    int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.npz_dir:
            self.npz_dir = os.path.join(self.project_root, "output_augmentation", "labels")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs_npz_gaussienne")
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - performances limitees a 1024x1024.")

cfg = TrainConfig()


In [ ]:
# ===================== CELLULE 4 — Dataset + dessin gaussienne (_draw_points_mask) =====================
class ECGNpzDataset(Dataset):
    # Dataset qui apparie chaque image augmentee P2 avec un mask binaire
    # genere a la volee depuis le NPZ (points d'intersection grille major).
    # Le mask est dessine en pleine resolution image (W,H native du NPZ),
    # puis redimensionne (NEAREST) a (img_width, img_height).

    def __init__(self, image_dir, npz_dir, npz_key="grid_major_5mm",
                 source_prefixes=None, img_height=1024, img_width=1024,
                 point_radius=3, gaussian_sigma=2.0, augment=False):
        self.image_dir = image_dir
        self.npz_dir   = npz_dir
        self.npz_key   = npz_key
        self.img_height, self.img_width = img_height, img_width
        self.point_radius   = point_radius     # fenetre (rayon) ou la gaussienne est dessinee
        self.gaussian_sigma = gaussian_sigma   # ecart-type de la gaussienne (px)

        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = os.path.splitext(fname)[0]
            npz_path = os.path.join(npz_dir, stem)
            labels_file = os.path.join(npz_path, "labels.npy.zst")
            if os.path.isfile(labels_file) and os.path.getsize(labels_file) > 0:
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "npz_path":   npz_path,
                    "stem":       stem,
                })

        # Augmentations couleur uniquement (la geometrie casserait l'alignement points/image)
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.ColorJitter(brightness=0.15, contrast=0.15,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees (key: {npz_key},"
              f" sources: {source_prefixes or 'toutes'})")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _draw_points_mask(pts_xy, H, W, radius, sigma=2.0):
        # Dessine des gaussiennes 2D centrees aux coordonnees pts_xy (Nx2) sur un fond noir HxW.
        # Chaque point recoit une gaussienne d'ecart-type 'sigma' (px), dessinee dans une fenetre
        # carree de rayon 'radius'. Pic au centre = 255 (apres normalisation), decroit progressivement.
        # Si plusieurs gaussiennes se chevauchent, on prend le MAX (pas la somme) pour eviter la
        # saturation et garder des pics independants.
        mask = np.zeros((H, W), dtype=np.float32)
        if len(pts_xy) == 0:
            return mask.astype(np.uint8)

        xs_f = pts_xy[:, 0]
        ys_f = pts_xy[:, 1]
        valid = (xs_f >= 0) & (xs_f < W) & (ys_f >= 0) & (ys_f < H)
        xs_f, ys_f = xs_f[valid], ys_f[valid]

        two_sigma2 = 2.0 * sigma * sigma
        for xf, yf in zip(xs_f, ys_f):
            xc, yc = int(round(xf)), int(round(yf))
            x0, x1 = max(0, xc - radius), min(W, xc + radius + 1)
            y0, y1 = max(0, yc - radius), min(H, yc + radius + 1)
            if x1 <= x0 or y1 <= y0:
                continue
            yy, xx = np.mgrid[y0:y1, x0:x1]
            # Distance au centre sub-pixel (on garde la precision du float)
            dx = xx - xf
            dy = yy - yf
            g  = np.exp(-(dx * dx + dy * dy) / two_sigma2)
            # Prendre le max pour les zones de chevauchement
            patch = mask[y0:y1, x0:x1]
            np.maximum(patch, g, out=patch)
            mask[y0:y1, x0:x1] = patch

        # Normaliser sur [0, 255] : pic gaussien atteint deja 1.0 au centre, on multiplie par 255
        return (mask * 255).clip(0, 255).astype(np.uint8)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # Image
        img = Image.open(s["image_path"]).convert("RGB")
        Wn, Hn = img.size  # taille native (avant resize)
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        # Mask depuis NPZ
        data = load_unified(s["npz_path"])
        pts = data.get(self.npz_key, np.empty((0, 2)))
        mask_full = self._draw_points_mask(pts, Hn, Wn, self.point_radius, self.gaussian_sigma)
        mask_pil = Image.fromarray(mask_full).resize(
            (self.img_width, self.img_height), Image.NEAREST
        )
        mask_np = np.array(mask_pil, dtype=np.float32) / 255.0

        # Augmentations couleur
        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = t["image"], t["mask"]

        img_tensor  = torch.from_numpy(img_np).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
# ===================== CELLULE 5 — Loss, metriques & boucle d'entrainement =====================
# ═══════════════ Loss ═══════════════
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        ps = torch.sigmoid(pred)
        inter = (ps * target).sum(dim=(2, 3))
        union = ps.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce, self.dice, self.bce_weight = nn.BCEWithLogitsLoss(), DiceLoss(), bce_weight
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + (1 - self.bce_weight) * self.dice(pred, target)

def get_loss(loss_type, bce_weight=0.5):
    return {"bce": nn.BCEWithLogitsLoss(),
            "dice": DiceLoss(),
            "bce_dice": BCEDiceLoss(bce_weight)}[loss_type]


# ═══════════════ Metrics ═══════════════
def compute_metrics(pred, target, threshold=0.5):
    with torch.no_grad():
        pb = (torch.sigmoid(pred) > threshold).float()
        inter = (pb * target).sum(dim=(2, 3))
        ps = pb.sum(dim=(2, 3)); ts = target.sum(dim=(2, 3))
        dice = (2*inter + 1e-6) / (ps + ts + 1e-6)
        iou  = (inter + 1e-6) / (ps + ts - inter + 1e-6)
        rec  = (inter + 1e-6) / (ts + 1e-6)
        prec = (inter + 1e-6) / (ps + 1e-6)
    return {"dice": dice.mean().item(), "iou": iou.mean().item(),
            "precision": prec.mean().item(), "recall": rec.mean().item()}


# ═══════════════ Visualisation ═══════════════
def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mt  = masks_true[i, 0].cpu().numpy()
        mp  = (torch.sigmoid(masks_pred[i, 0]).cpu().numpy() > 0.5).astype(float)
        axes[i, 0].imshow(img);                  axes[i, 0].set_title("Image augmentee", fontsize=9); axes[i, 0].axis("off")
        axes[i, 1].imshow(mt, cmap="gray", vmin=0, vmax=1); axes[i, 1].set_title("Points GT (5mm)", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(mp, cmap="gray", vmin=0, vmax=1); axes[i, 2].set_title("Points predits", fontsize=9); axes[i, 2].axis("off")
        ov = img.copy()
        tp = (mp > 0.5) & (mt > 0.5); fp = (mp > 0.5) & (mt < 0.5); fn = (mp < 0.5) & (mt > 0.5)
        ov[tp] = [0, 1, 0]; ov[fp] = [1, 0, 0]; ov[fn] = [0, 0, 1]
        axes[i, 3].imshow(ov); axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9); axes[i, 3].axis("off")
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches="tight"); plt.close()


# ═══════════════ Boucles d'entrainement ═══════════════
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.3f}")
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}


def _plot_training_curves(history, save_path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    e = range(1, len(history["train_loss"]) + 1)
    a1.plot(e, history["train_loss"], "b-", label="Train")
    a1.plot(e, history["val_loss"],   "r-", label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e, history["train_dice"], "b-",  label="Train Dice")
    a2.plot(e, history["val_dice"],   "r-",  label="Val Dice")
    a2.plot(e, history["train_iou"],  "b--", alpha=0.5, label="Train IoU")
    a2.plot(e, history["val_iou"],    "r--", alpha=0.5, label="Val IoU")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Dice & IoU"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close()


def train(cfg: TrainConfig):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    cfg_dict = {k: (v if isinstance(v, (int, float, bool, list, type(None))) else str(v))
                for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg_dict, f, indent=2)

    print(f"\n{'='*60}\n  Entrainement U-Net -- Detection points intersections (NPZ)\n{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Cible NPZ   : {cfg.npz_key}  (point_radius={cfg.point_radius}px, sigma={cfg.gaussian_sigma}px - GAUSSIAN)")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}\n{'='*60}\n")

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = torch.device(cfg.device)

    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  NPZ   : {cfg.npz_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                             cfg.train_sources, cfg.img_height, cfg.img_width,
                             cfg.point_radius, cfg.gaussian_sigma, augment=True)
    print(f"  Val   (sources: {cfg.val_sources}):")
    val_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                           cfg.val_sources, cfg.img_height, cfg.img_width,
                           cfg.point_radius, cfg.gaussian_sigma, augment=False)

    if len(train_ds) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee."); return

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

    print("\n[*] Construction du modele...")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=cfg.encoder_weights,
                     in_channels=cfg.in_channels, classes=cfg.num_classes, activation=None).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Parametres: {total:,}")

    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience, factor=cfg.scheduler_factor)

    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "train_iou", "val_iou", "lr"]}
    best_val_dice, no_improve = 0, 0

    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(1, cfg.num_epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]

        tl, tm = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = validate(model, val_loader, criterion, device)
        scheduler.step(vl)

        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_dice"].append(tm["dice"]); history["val_dice"].append(vm["dice"])
        history["train_iou"].append(tm["iou"]);   history["val_iou"].append(vm["iou"])
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
              f"Train Loss: {tl:.4f}  Dice: {tm['dice']:.3f} | "
              f"Val Loss: {vl:.4f}  Dice: {vm['dice']:.3f}  IoU: {vm['iou']:.3f} | "
              f"LR: {lr:.1e} | {time.time()-t0:.1f}s")

        if vm["dice"] > best_val_dice:
            best_val_dice = vm["dice"]; no_improve = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": best_val_dice, "config": cfg_dict},
                       os.path.join(run_dir, "checkpoints", "best_model.pth"))
            print(f"  [BEST] Nouveau meilleur modele (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1

        if epoch % cfg.save_every_n_epochs == 0:
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": vm["dice"]},
                       os.path.join(run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"))

        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                si, sm = next(iter(val_loader))
                sp = model(si.to(device)).cpu()
                save_prediction_grid(si, sm, sp,
                    os.path.join(run_dir, "visualizations", f"epoch_{epoch:03d}.png"))

        if no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration"); break

    total_time = time.time() - t_start
    print(f"\n{'='*60}\n  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}\n{'='*60}")
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))
    return run_dir


In [ ]:
# ===================== CELLULE 6 — Lancement de l'entrainement =====================
# Lancement de l'entrainement
train(cfg)


In [ ]:
# ===================== CELLULE 7 — Chargement du modele entraine =====================
import glob
from IPython.display import display

runs_dir = cfg.output_dir
run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"Aucun run trouve dans {runs_dir}")
run_dir = run_dirs[-1]
print(f"Run selectionne : {run_dir}")

best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
ckpt_list = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))
checkpoint_path = best_path if os.path.exists(best_path) else (ckpt_list[-1] if ckpt_list else None)
if checkpoint_path is None:
    raise FileNotFoundError("Aucun checkpoint trouve")
print(f"Checkpoint : {checkpoint_path}")

DEVICE = cfg.device
model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                 in_channels=cfg.in_channels, classes=cfg.num_classes)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"Modele charge - epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")


In [ ]:
# ===================== CELLULE 8 — Liste des images train/val =====================
image_dir = cfg.image_dir
val_images   = sorted([f for f in os.listdir(image_dir)
                       if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir)
                       if (f.startswith("ECG_031") or f.startswith("ECG_032"))
                       and f.endswith(".webp")])
print(f"Train: {len(train_images)} | Val: {len(val_images)}")


In [ ]:
# ===================== CELLULE 9 — Visu complete : image + heatmap + zoom + erreurs (par point GT) + zooms blobs =====================
import cv2
from scipy.spatial import cKDTree
from matplotlib.patches import Patch
%matplotlib inline

SET, INDEX           = "val", 8
DEV_TOL              = 5    # px : deviation max pour TP. Au-dela -> FAUX NEGATIF (detecte mais devie).
MAX_MATCH_DIST       = 15   # px : distance max pour APPARIER une prediction a un point GT.
SEPARATION_THRESHOLD = 0.3  # seuil bas pour separer les blobs predits
MIN_BLOB_AREA        = 3
MERGE_DIST           = 25   # px : fusionne 2 detections plus proches que ca (meme intersection coupee en 2 par un pli)
DISK_R               = 7    # rayon (px) des disques de la carte d'erreurs (lisibilite plein-cadre)

image_list  = val_images if SET == "val" else train_images
sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
npz_path    = os.path.join(cfg.npz_dir, sample_file.replace(".webp", ""))
print(f"Image : {sample_file}")

# --- Image native + version modele ---
img_pil    = Image.open(sample_path).convert("RGB")
Wn, Hn     = img_pil.size
img_native = np.array(img_pil)
img_model  = np.array(img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR), np.float32) / 255.0

# --- Heatmap gaussienne CIBLE complete (tous les globes, sigma=2) ---
data    = load_unified(npz_path, load_maps=False)
gt_full = data.get(cfg.npz_key, np.empty((0, 2)))
gt_full = gt_full[np.isfinite(gt_full).all(1)] if len(gt_full) else gt_full
hm_full = ECGNpzDataset._draw_points_mask(gt_full, Hn, Wn, cfg.point_radius, cfg.gaussian_sigma).astype(np.float32) / 255.0
cyz, cxz = Hn // 2, Wn // 2; Zr = 200

# --- GT valides (dans l image) ---
if len(gt_full):
    vv = (gt_full[:, 0] >= 0) & (gt_full[:, 0] < Wn) & (gt_full[:, 1] >= 0) & (gt_full[:, 1] < Hn)
    gt_pts = gt_full[vv]
else:
    gt_pts = gt_full

# --- Inference -> resolution native ---
with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(torch.from_numpy(img_model).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE))).squeeze().cpu().numpy()
pred_native = cv2.resize(pred_sigmoid, (Wn, Hn), interpolation=cv2.INTER_LINEAR)

# --- Masque SIGNAL (modele dedie) : sert d'ARBITRE pour les doublons (candidat sur le trace -> rejete) ---
import glob as _glob
_sig_runs = sorted(_glob.glob(os.path.join(cfg.project_root, "training", "runs_signal", "run_*", "checkpoints", "best_model.pth")))
signal_native = None
if _sig_runs:
    if "sig_model" not in globals():                   # charge une seule fois (293 Mo) puis met en cache
        sig_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
        _sd = torch.load(_sig_runs[-1], map_location=DEVICE); _sd = _sd.get("model_state_dict", _sd) if isinstance(_sd, dict) else _sd
        sig_model.load_state_dict(_sd); sig_model.eval()
        print(f"Modele signal (arbitre) : {os.path.relpath(_sig_runs[-1], cfg.project_root)}")
    with torch.no_grad():
        _sig = torch.sigmoid(sig_model(torch.from_numpy(img_model).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE))).squeeze().cpu().numpy()
    signal_native = cv2.resize(_sig, (Wn, Hn), interpolation=cv2.INTER_LINEAR)   # proba "signal" par pixel
else:
    print("Modele signal absent -> arbitrage par max-confiance seulement")

# --- Centres predits via centroide pondere (sub-pixel) ---
sep = (pred_native > SEPARATION_THRESHOLD).astype(np.uint8)
nlab, labels, stats, _ = cv2.connectedComponentsWithStats(sep, connectivity=8)
pred_list, pred_conf, pred_labs = [], [], []
for lab in range(1, nlab):
    if stats[lab, cv2.CC_STAT_AREA] < MIN_BLOB_AREA:
        continue
    m = labels == lab; w = pred_native[m]; yy, xx = np.where(m); ws = w.sum()
    if ws < 1e-6:
        continue
    pred_list.append((float((xx * w).sum() / ws), float((yy * w).sum() / ws)))
    pred_conf.append(float(w.max())); pred_labs.append(lab)
pred_pts  = np.asarray(pred_list) if pred_list else np.empty((0, 2))
pred_conf = np.asarray(pred_conf)
pred_labs = np.asarray(pred_labs)

# --- Fusion des detections en DOUBLE (blob coupe par un pli OU faux pic la ou le signal croise la grille) ---
# ARBITRE = le modele SIGNAL : entre 2 candidats proches, on garde celui le MOINS sur le trace
# (le faux pic est sur le signal). Egalite (les 2 hors-signal) -> on tranche par max-confiance.
# NB : arbitrage sur les DOUBLONS seulement ; une intersection isolee sous le signal n'est pas un
# doublon, donc jamais supprimee (on ne perd pas les vraies intersections sous le trace).
def _sig_score(p):
    if signal_native is None:
        return 0.0
    x, y = int(p[0]), int(p[1])
    return float(signal_native[max(0, y - 3):y + 4, max(0, x - 3):x + 4].mean())

def _merge_close(pts, confs, dmin):
    if len(pts) == 0:
        return pts
    tree = cKDTree(pts); used = np.zeros(len(pts), bool); keep = []
    for i in np.argsort(-confs):
        if used[i]:
            continue
        idxs = [j for j in tree.query_ball_point(pts[i], dmin) if not used[j]]
        used[idxs] = True
        if len(idxs) == 1:
            keep.append(pts[idxs[0]]); continue
        sscore = np.array([_sig_score(pts[j]) for j in idxs])
        cand = [j for j, sc in zip(idxs, sscore) if sc <= sscore.min() + 0.1]   # candidats les MOINS sur le signal
        keep.append(pts[cand[int(np.argmax(confs[cand]))]])                     # parmi eux, le + confiant
    return np.asarray(keep)

n_before = len(pred_pts)
pred_pts = _merge_close(pred_pts, pred_conf, MERGE_DIST)
print(f"Detections : {n_before} -> {len(pred_pts)} apres fusion des doublons (< {MERGE_DIST} px)")

# --- Matching greedy : pour CHAQUE point GT, sa deviation au plus proche predit (NaN si non apparie) ---
nb_gt, nb_pred = len(gt_pts), len(pred_pts)
gt_dev = np.full(nb_gt, np.nan)            # deviation par point GT
mp, mg, md = [], [], []                    # paires appariees (pour les zooms)
if nb_gt and nb_pred:
    tree = cKDTree(gt_pts); dists, gidx = tree.query(pred_pts, k=1)
    used = set()
    for pi in np.argsort(dists):
        d, gi = dists[pi], int(gidx[pi])
        if d <= MAX_MATCH_DIST and gi not in used:
            gt_dev[gi] = d; used.add(gi)
            mp.append(pi); mg.append(gi); md.append(d)
matched    = ~np.isnan(gt_dev)
arr_d      = np.asarray(md) if md else np.array([])
match_gt   = gt_pts[mg]   if mg else np.empty((0, 2))
match_pred = pred_pts[mp] if mp else np.empty((0, 2))

# --- 3 categories DISJOINTES au niveau du point GT (aucune superposition possible) ---
tp_pts = gt_pts[matched & (gt_dev <= DEV_TOL)]   # bien detecte
fn_pts = gt_pts[matched & (gt_dev >  DEV_TOL)]   # FAUX NEGATIF : detecte mais devie
vn_pts = gt_pts[~matched]                        # VRAI NEGATIF : totalement manque
nb_tp, nb_fn, nb_vn = len(tp_pts), len(fn_pts), len(vn_pts)
nb_fp_extra = nb_pred - len(mp)                  # predictions sans aucun GT (info)
recall = nb_tp / nb_gt if nb_gt else 0.0

print("=" * 64)
print(f"  GT={nb_gt}   Pred={nb_pred}")
print(f"  TP (bien detecte)            = {nb_tp}   ({100*nb_tp/max(nb_gt,1):.1f}%)")
print(f"  FN (detecte mais devie>{DEV_TOL}px) = {nb_fn}   ({100*nb_fn/max(nb_gt,1):.1f}%)")
print(f"  VN (intersection manquee)    = {nb_vn}   ({100*nb_vn/max(nb_gt,1):.1f}%)")
print(f"  FP (prediction sans GT)      = {nb_fp_extra}")
if len(arr_d):
    print(f"[DEVIATION PIXEL]  moyenne={arr_d.mean():.2f}  mediane={np.median(arr_d):.2f}  p95={np.percentile(arr_d, 95):.2f}  max={arr_d.max():.2f} px")
print("=" * 64)

# disques pleins (1 par point) -> carte categorielle nette, sans superposition
def _disks(pts, H, W, r):
    m = np.zeros((H, W), np.float32)
    for x, y in pts:
        cv2.circle(m, (int(round(x)), int(round(y))), r, 1.0, -1)
    return m

# carte d'erreurs : fond sombre (TN), TP vert faible, FN orange, VN rouge
err = (img_native.astype(np.float32) / 255.0) * 0.15
g = _disks(tp_pts, Hn, Wn, DISK_R); err[..., 1] = np.clip(err[..., 1] + 0.45 * g, 0, 1)                       # vert = TP
f = _disks(fn_pts, Hn, Wn, DISK_R); err[..., 0] = np.clip(err[..., 0] + f, 0, 1); err[..., 1] = np.clip(err[..., 1] + 0.6 * f, 0, 1)  # orange = FN
r = _disks(vn_pts, Hn, Wn, DISK_R); err[..., 0] = np.clip(err[..., 0] + r, 0, 1)                              # rouge = VN

# === FIGURE 1 : image / heatmap / zoom / carte d'erreurs (par point GT) ===
fig, ax = plt.subplots(1, 4, figsize=(26, 7))
fig.suptitle(f"[{SET.upper()}] {sample_file}  -  sigma={cfg.gaussian_sigma}  -  recall={recall*100:.1f}%  dev.moy={arr_d.mean() if len(arr_d) else float('nan'):.2f}px", fontsize=13)
ax[0].imshow(img_native); ax[0].set_title("Image originale"); ax[0].axis("off")
ax[1].imshow(hm_full, cmap="hot", vmin=0, vmax=1); ax[1].set_title(f"Heatmap gaussienne CIBLE ({len(gt_full)} globes)"); ax[1].axis("off")
ax[1].add_patch(plt.Rectangle((cxz - Zr, cyz - Zr), 2 * Zr, 2 * Zr, fill=False, ec="lime", lw=2))
ax[2].imshow(hm_full[cyz - Zr:cyz + Zr, cxz - Zr:cxz + Zr], cmap="hot", vmin=0, vmax=1); ax[2].set_title("Zoom : globes ronds bien separes"); ax[2].axis("off")
ax[3].imshow(err)
ax[3].set_title(f"Erreurs/point GT : TP={nb_tp} vert | FN devie={nb_fn} orange | VN manque={nb_vn} rouge", fontsize=10)
ax[3].legend(handles=[Patch(color=(0, 0.45, 0), label="Bien detecte (TP)"),
                      Patch(color=(1, 0.6, 0), label=f"Faux negatif : devie > {DEV_TOL}px"),
                      Patch(color=(1, 0, 0), label="Vrai negatif : manque (rien detecte)")],
             loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=1, fontsize=8, framealpha=0.9)
ax[3].axis("off")
plt.tight_layout(); plt.show(); plt.close()

# === Resultat PUR de la detection de la grille (intersections predites, SANS TP/FN/FP) ===
det = img_native.copy()
for x, y in pred_pts:
    cv2.circle(det, (int(round(x)), int(round(y))), 5, (0, 255, 255), -1)   # cyan = intersection detectee
fig2, ax2 = plt.subplots(1, 2, figsize=(24, 9))
fig2.suptitle(f"Detection grille (resultat brut) : {len(pred_pts)} intersections detectees", fontsize=14)
ax2[0].imshow(det); ax2[0].set_title("Image + intersections detectees (cyan)"); ax2[0].axis("off")
ax2[0].add_patch(plt.Rectangle((cxz - Zr, cyz - Zr), 2 * Zr, 2 * Zr, fill=False, ec="lime", lw=2))
ax2[1].imshow(det[cyz - Zr:cyz + Zr, cxz - Zr:cxz + Zr]); ax2[1].set_title("Zoom"); ax2[1].axis("off")
plt.tight_layout(); plt.show(); plt.close()


In [ ]:
# ===================== CELLULE 10 — Inference sur image reelle =====================
# Test du modele sur image reelle (data/output_real/)
import os, glob, cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp

%matplotlib inline

# Verification stricte : comparer les poids en memoire vs disk
import glob, os, torch
runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
ckpt_disk = torch.load(ckpt_path, map_location='cpu', weights_only=False)

# Recuperer un poids specifique en memoire vs disk
weight_name = list(ckpt_disk["model_state_dict"].keys())[5]   # un poids arbitraire
w_mem  = model.state_dict()[weight_name].cpu().flatten()[:5]
w_disk = ckpt_disk["model_state_dict"][weight_name].flatten()[:5]
print(f"Weight tested      : {weight_name}")
print(f"In memory (5 vals) : {w_mem.numpy()}")
print(f"On disk    (5 vals): {w_disk.numpy()}")
print(f"MATCH              : {torch.allclose(w_mem, w_disk)}")
print(f"Ckpt epoch         : {ckpt_disk['epoch']}")
print(f"Ckpt val_dice      : {ckpt_disk.get('val_dice', 'N/A')}")

# === A modifier au besoin ===
REAL_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main\data\output_real"
SUBFOLDER = "photos_crumbles"   # ou None pour scanner tout
INDEX = 1                        # quel image dans la liste
OVERLAY_COLOR = (255, 0, 0)
OVERLAY_ALPHA = 0.55

# === Liste des images reelles ===
if SUBFOLDER:
    search = os.path.join(REAL_ROOT, SUBFOLDER, "*")
else:
    search = os.path.join(REAL_ROOT, "**", "*")
real_files = sorted([f for f in glob.glob(search, recursive=True)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
print(f"{len(real_files)} images reelles trouvees")

if INDEX >= len(real_files):
    raise IndexError(f"INDEX={INDEX} hors limites (max {len(real_files)-1})")
img_path = real_files[INDEX]
print(f"Image : {img_path}")

# === Chargement du modele (si pas deja en memoire) ===
try:
    model
except NameError:
    runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
    if not runs:
        raise FileNotFoundError(f"Aucun run dans {cfg.output_dir}")
    ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                     in_channels=cfg.in_channels, classes=cfg.num_classes)
    ckpt = torch.load(ckpt_path, map_location=cfg.device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(cfg.device).eval()
    print(f"Modele charge - epoch {ckpt['epoch']} | val_dice = {ckpt.get('val_dice', 'N/A')}")

# === Preprocessing (identique au training : /255, pas d'imagenet norm) ===
img_pil = Image.open(img_path).convert("RGB")
W_orig, H_orig = img_pil.size
print(f"Resolution native : {W_orig}x{H_orig}")

img_resized = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np = np.array(img_resized, dtype=np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(cfg.device)

# === Inference ===
with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()

pred_full = cv2.resize(pred_sigmoid, (W_orig, H_orig), interpolation=cv2.INTER_LINEAR)
pred_binary = (pred_full > 0.5).astype(np.uint8) * 255

print(f"Heatmap : max={pred_sigmoid.max():.3f}  mean={pred_sigmoid.mean():.4f}")
print(f"Pixels positifs (seuil 0.5) : {(pred_binary>0).sum()} ({100*(pred_binary>0).sum()/pred_binary.size:.2f}%)")

# === Superposition ===
img_arr = np.array(img_pil)
overlay = img_arr.copy().astype(np.float32)
mask_bool = pred_binary > 127
color = np.array(OVERLAY_COLOR, dtype=np.float32)
overlay[mask_bool] = (1 - OVERLAY_ALPHA) * overlay[mask_bool] + OVERLAY_ALPHA * color
overlay = overlay.clip(0, 255).astype(np.uint8)

# === Affichage 4-up ===
fig, axes = plt.subplots(1, 4, figsize=(28, 8))
fig.suptitle(f"Inference reelle : {os.path.basename(img_path)}", fontsize=12)
axes[0].imshow(img_arr);                                                          axes[0].set_title(f"Image reelle ({W_orig}x{H_orig})");           axes[0].axis("off")
axes[1].imshow(pred_full, cmap="hot", vmin=0, vmax=max(pred_full.max(), 1e-6));   axes[1].set_title(f"Sigmoid (max={pred_full.max():.2f})");        axes[1].axis("off")
axes[2].imshow(pred_binary, cmap="gray", vmin=0, vmax=255);                       axes[2].set_title("Mask binarise (seuil 0.5)");                   axes[2].axis("off")
axes[3].imshow(overlay);                                                          axes[3].set_title("Superposition");                               axes[3].axis("off")
plt.tight_layout(); plt.show()

# ===================== Detection des intersections : decodage + dedoublonnage (arbitre-signal) =====================
from scipy.spatial import cKDTree
SEP_TH_R, MIN_AREA_R = 0.3, 3

# --- centres predits (centroide pondere) + confiance ---
nlab, labels_r, stats_r, _ = cv2.connectedComponentsWithStats((pred_full > SEP_TH_R).astype(np.uint8), 8)
pl, pc = [], []
for lab in range(1, nlab):
    if stats_r[lab, cv2.CC_STAT_AREA] < MIN_AREA_R:
        continue
    x0, y0 = stats_r[lab, cv2.CC_STAT_LEFT], stats_r[lab, cv2.CC_STAT_TOP]
    ww, hh = stats_r[lab, cv2.CC_STAT_WIDTH], stats_r[lab, cv2.CC_STAT_HEIGHT]
    sL = labels_r[y0:y0+hh, x0:x0+ww]; sP = pred_full[y0:y0+hh, x0:x0+ww]
    m = sL == lab; w = sP[m]; yy, xx = np.where(m); ws = w.sum()
    if ws < 1e-6:
        continue
    pl.append((x0 + (xx*w).sum()/ws, y0 + (yy*w).sum()/ws)); pc.append(float(w.max()))
det_pts = np.asarray(pl) if pl else np.empty((0, 2)); det_conf = np.asarray(pc)

# --- seuil de fusion ADAPTATIF : espacement estime depuis les predictions (echelles variees) ---
if len(det_pts) > 5:
    spacing = float(np.median(cKDTree(det_pts).query(det_pts, k=2)[0][:, 1]))
    MERGE_DIST_R = 0.4 * spacing
else:
    spacing, MERGE_DIST_R = float("nan"), 25.0

# --- masque SIGNAL (arbitre) sur l image reelle ---
_sig_runs = sorted(glob.glob(os.path.join(cfg.project_root, "training", "runs_signal", "run_*", "checkpoints", "best_model.pth")))
signal_real = None
if _sig_runs:
    if "sig_model" not in globals():
        sig_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(cfg.device)
        _sd = torch.load(_sig_runs[-1], map_location=cfg.device, weights_only=False)
        _sd = _sd.get("model_state_dict", _sd) if isinstance(_sd, dict) else _sd
        sig_model.load_state_dict(_sd); sig_model.eval()
        print(f"Modele signal (arbitre) : {os.path.relpath(_sig_runs[-1], cfg.project_root)}")
    with torch.no_grad():
        _sr = torch.sigmoid(sig_model(img_tensor)).squeeze().cpu().numpy()
    signal_real = cv2.resize(_sr, (W_orig, H_orig), interpolation=cv2.INTER_LINEAR)
else:
    print("Modele signal absent -> arbitrage par max-confiance seulement")

def _sig_score_r(p):
    if signal_real is None:
        return 0.0
    x, y = int(p[0]), int(p[1])
    return float(signal_real[max(0, y-3):y+4, max(0, x-3):x+4].mean())

def _dedup(pts, confs, dmin):
    if len(pts) == 0:
        return pts
    tree = cKDTree(pts); used = np.zeros(len(pts), bool); keep = []
    for i in np.argsort(-confs):
        if used[i]:
            continue
        idxs = [j for j in tree.query_ball_point(pts[i], dmin) if not used[j]]
        used[idxs] = True
        if len(idxs) == 1:
            keep.append(pts[idxs[0]]); continue
        sc = np.array([_sig_score_r(pts[j]) for j in idxs])
        cand = [j for j, ssc in zip(idxs, sc) if ssc <= sc.min() + 0.1]   # candidats les moins sur le signal
        keep.append(pts[cand[int(np.argmax(confs[cand]))]])              # parmi eux, le + confiant
    return np.asarray(keep)

n0 = len(det_pts); det_pts = _dedup(det_pts, det_conf, MERGE_DIST_R)
print(f"Intersections : {n0} -> {len(det_pts)} apres dedoublonnage "
      f"(espacement~{spacing:.0f}px, seuil {MERGE_DIST_R:.0f}px, arbitre-signal={'oui' if signal_real is not None else 'non'})")

# --- affichage : detection pure (intersections detectees, sans classification) ---
det_img = img_arr.copy()
for x, y in det_pts:
    cv2.circle(det_img, (int(round(x)), int(round(y))), 5, (0, 255, 255), -1)
cyr, cxr = H_orig // 2, W_orig // 2; Zr2 = min(220, W_orig // 4, H_orig // 4)
fig2, ax2 = plt.subplots(1, 2, figsize=(22, 9))
fig2.suptitle(f"Detection grille (decodee + dedoublonnee) : {len(det_pts)} intersections", fontsize=13)
ax2[0].imshow(det_img); ax2[0].set_title("Image + intersections detectees (cyan)"); ax2[0].axis("off")
ax2[0].add_patch(plt.Rectangle((cxr - Zr2, cyr - Zr2), 2 * Zr2, 2 * Zr2, fill=False, ec="lime", lw=2))
ax2[1].imshow(det_img[cyr - Zr2:cyr + Zr2, cxr - Zr2:cxr + Zr2]); ax2[1].set_title("Zoom"); ax2[1].axis("off")
plt.tight_layout(); plt.show()